In [1]:
!pip install stable-baselines3[extra]
!pip install gymnasium
!pip install pygame
!pip install transformers
!pip install torch


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 56.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 20.6 MB/s eta 0:00:00


In [2]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np

class DisasterEnv(gym.Env):
    def __init__(self):
        super(DisasterEnv, self).__init__()
        self.grid_size = 10
        self.action_space = spaces.Discrete(4)
        self.observation_space = spaces.Box(
            low=-1, high=4, shape=(self.grid_size, self.grid_size), dtype=np.int32
        )
        self.max_steps = 200
        self.civilians_to_save = 2

    def update_memory(self):
        vision_range = 2
        explorers = [self.agent_pos, self.drone_pos]
        for ex, ey in explorers:
            for dx in range(-vision_range, vision_range + 1):
                for dy in range(-vision_range, vision_range + 1):
                    world_x, world_y = ex + dx, ey + dy
                    if 0 <= world_x < self.grid_size and 0 <= world_y < self.grid_size:
                        self.memory_grid[world_x][world_y] = self.grid[world_x][world_y]

    def get_local_observation(self):
        return self.memory_grid.copy()

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.current_step = 0
        self.civilians_saved = 0
        self.grid = np.zeros((self.grid_size, self.grid_size), dtype=np.int32)
        self.memory_grid = np.full((self.grid_size, self.grid_size), -1, dtype=np.int32)

        self.grid[2][3] = 1
        self.grid[7][8] = 1
        self.grid[5][5] = 2
        self.grid[1][8] = 2

        self.agent_pos = [0, 0]
        self.drone_pos = [9, 9]
        self.grid[self.agent_pos[0]][self.agent_pos[1]] = 3
        self.grid[self.drone_pos[0]][self.drone_pos[1]] = 4
        self.tile_under_agent = 0
        self.tile_under_drone = 0

        self.messages = []
        self.reported_locations = set()

        self.explored_cells = set()
        self.discovery_reward = 0

        self.update_memory()
        return self.get_local_observation(), {}

    def spread_fire(self):
        new_fires = []
        for row in range(self.grid_size):
            for col in range(self.grid_size):
                if self.grid[row][col] == 1:
                    neighbors = [(-1, 0), (1, 0), (0, -1), (0, 1)]
                    for dx, dy in neighbors:
                        nx, ny = row + dx, col + dy
                        if 0 <= nx < self.grid_size and 0 <= ny < self.grid_size:
                            if self.grid[nx][ny] == 0:
                                if self.np_random.random() < 0.10:
                                    new_fires.append((nx, ny))
        for x, y in new_fires:
            self.grid[x][y] = 1

    def move_drone(self):
        self.grid[self.drone_pos[0]][self.drone_pos[1]] = self.tile_under_drone
        action = self.np_random.integers(0, 4)
        new_x, new_y = self.drone_pos

        if action == 0 and new_x > 0: new_x -= 1
        elif action == 1 and new_x < self.grid_size - 1: new_x += 1
        elif action == 2 and new_y > 0: new_y -= 1
        elif action == 3 and new_y < self.grid_size - 1: new_y += 1

        self.drone_pos = [new_x, new_y]
        self.tile_under_drone = self.grid[new_x][new_y]
        self.grid[new_x][new_y] = 4

    def generate_drone_message(self):
        drone_x, drone_y = self.drone_pos
        vision_range = 1
        for dx in range(-vision_range, vision_range + 1):
            for dy in range(-vision_range, vision_range + 1):
                world_x, world_y = drone_x + dx, drone_y + dy
                if 0 <= world_x < self.grid_size and 0 <= world_y < self.grid_size:
                    tile = self.grid[world_x][world_y]
                    location = (world_x, world_y)

                    if location not in self.reported_locations:
                        if tile == 1:
                            self.messages.append(f"Fire detected at ({world_x}, {world_y})")
                            self.reported_locations.add(location)
                        elif tile == 2:
                            self.messages.append(f"Civilian detected at ({world_x}, {world_y})")
                            self.reported_locations.add(location)
                            self.discovery_reward += 20

    def step(self, action):
        self.current_step += 1

        reward = -1
        done = False
        truncated = False

        self.grid[self.agent_pos[0]][self.agent_pos[1]] = self.tile_under_agent
        new_x, new_y = self.agent_pos

        if action == 0 and new_x > 0: new_x -= 1
        elif action == 1 and new_x < self.grid_size - 1: new_x += 1
        elif action == 2 and new_y > 0: new_y -= 1
        elif action == 3 and new_y < self.grid_size - 1: new_y += 1

        self.agent_pos = [new_x, new_y]
        self.tile_under_agent = self.grid[new_x][new_y]

        current_cell = (self.agent_pos[0], self.agent_pos[1])
        if current_cell not in self.explored_cells:
            reward += 5
            self.explored_cells.add(current_cell)

        if self.tile_under_agent == 2:
            reward += 100
            self.civilians_saved += 1
            self.tile_under_agent = 0

            if self.civilians_saved == self.civilians_to_save:
                reward += 200
                done = True

        elif self.tile_under_agent == 1:
            reward -= 100

        self.grid[self.agent_pos[0]][self.agent_pos[1]] = 3

        self.move_drone()
        self.generate_drone_message()

        reward += self.discovery_reward
        self.discovery_reward = 0

        if self.current_step % 5 == 0:
            self.spread_fire()

        if self.current_step >= self.max_steps:
            truncated = True

        self.update_memory()

        return self.get_local_observation(), reward, done, truncated, {}

    def render(self):
        print(self.grid)

In [3]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from gymnasium.wrappers import FlattenObservation

# Base env for training
env = DisasterEnv()
wrapped_env = FlattenObservation(env)

# Initialize PPO
model = PPO("MlpPolicy", wrapped_env, verbose=1, device="auto")

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetim

In [4]:
import os
from stable_baselines3.common.callbacks import EvalCallback

# Setup logs and evaluation environment
log_dir = "./logs/"
os.makedirs(log_dir, exist_ok=True)

eval_env = FlattenObservation(DisasterEnv())
eval_env = Monitor(eval_env)

# Save the best model every 5k steps
eval_callback = EvalCallback(
    eval_env,
    best_model_save_path=log_dir,
    log_path=log_dir,
    eval_freq=5000,
    deterministic=True,
    render=False
)

print("=== Starting 200,000 Step Training ===")
model.learn(total_timesteps=200000, callback=eval_callback)

=== Starting 200,000 Step Training ===
----------------------------------
| rollout/           |           |
|    ep_len_mean     | 192       |
|    ep_rew_mean     | -7.41e+03 |
| time/              |           |
|    fps             | 318       |
|    iterations      | 1         |
|    time_elapsed    | 6         |
|    total_timesteps | 2048      |
----------------------------------
------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 190          |
|    ep_rew_mean          | -7.36e+03    |
| time/                   |              |
|    fps                  | 234          |
|    iterations           | 2            |
|    time_elapsed         | 17           |
|    total_timesteps      | 4096         |
| train/                  |              |
|    approx_kl            | 0.0034182444 |
|    clip_fraction        | 0            |
|    clip_range           | 0.2          |
|    entropy_loss         | -1.38        |
|    expl

In [6]:
from transformers import pipeline
import torch

print("Loading Hugging Face Transformer...")

# Check if Colab GPU is actually enabled, otherwise fallback to CPU
device_id = 0 if torch.cuda.is_available() else -1

nlp_brain = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=device_id  # <--- FIX: Uses 0 for GPU, or -1 for CPU
)
print("Transformer Loaded and Ready!")

Loading Hugging Face Transformer...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Transformer Loaded and Ready!


In [7]:
import time

print("=== Loading Best Brain ===")
best_model = PPO.load("./logs/best_model", env=wrapped_env)

obs, info = wrapped_env.reset()
done = False
truncated = False
step_counter = 0
total_reward = 0
printed_messages_count = 0

candidate_labels = ["Critical Rescue Target", "Lethal Hazard", "Routine Exploration"]

print("=== STARTING AI-ENHANCED RESCUE MISSION ===")

while not done and not truncated:
    # 1. RL Agent Moves
    action, _states = best_model.predict(obs, deterministic=True)
    obs, reward, done, truncated, info = wrapped_env.step(action)

    step_counter += 1
    total_reward += reward

    # 2. Check Communications
    current_messages = wrapped_env.unwrapped.messages
    if len(current_messages) > printed_messages_count:
        new_messages = current_messages[printed_messages_count:]

        for msg in new_messages:
            # 3. HUGGING FACE NLP CLASSIFICATION
            analysis = nlp_brain(msg, candidate_labels)
            top_category = analysis['labels'][0]
            confidence = analysis['scores'][0] * 100

            print(f"\n[Step {step_counter}] 🚁 DRONE ALERT: {msg}")

            if top_category == "Lethal Hazard":
                print(f"   ⚠️ NLP CLASSIFICATION: {top_category} ({confidence:.1f}% confidence)")
                print(f"   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.")
            elif top_category == "Critical Rescue Target":
                print(f"   🚨 NLP CLASSIFICATION: {top_category} ({confidence:.1f}% confidence)")
                print(f"   🤖 SYSTEM COMMAND: High priority! Pinging coordinates to Rescue Agent.")
            else:
                print(f"   ℹ️ NLP CLASSIFICATION: {top_category} ({confidence:.1f}% confidence)")

        printed_messages_count = len(current_messages)

print(f"\n=== MISSION COMPLETE ===")
print(f"Steps Taken: {step_counter}")
print(f"Total Reward Gained: {total_reward}")

=== Loading Best Brain ===
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
=== STARTING AI-ENHANCED RESCUE MISSION ===


/usr/local/lib/python3.12/dist-packages/stable_baselines3/common/on_policy_algorithm.py:150: UserWarning: You are trying to run PPO on the GPU, but it is primarily intended to run on the CPU when not using a CNN policy (you are using ActorCriticPolicy which should be a MlpPolicy). See https://github.com/DLR-RM/stable-baselines3/issues/1245 for more info. You can pass `device='cpu'` or `export CUDA_VISIBLE_DEVICES=` to force using the CPU.Note: The model will train, but the GPU utilization will be poor and the training might take longer than on CPU.
  warnings.warn(



[Step 3] 🚁 DRONE ALERT: Fire detected at (7, 8)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (48.3% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 22] 🚁 DRONE ALERT: Civilian detected at (1, 8)
   🚨 NLP CLASSIFICATION: Critical Rescue Target (42.0% confidence)
   🤖 SYSTEM COMMAND: High priority! Pinging coordinates to Rescue Agent.

[Step 53] 🚁 DRONE ALERT: Fire detected at (6, 7)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (49.2% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 53] 🚁 DRONE ALERT: Fire detected at (6, 8)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (50.8% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 62] 🚁 DRONE ALERT: Fire detected at (5, 7)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (50.8% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 64] 🚁 DRONE ALERT: Fire detected at (7, 9)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (49.4% confidence

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



[Step 70] 🚁 DRONE ALERT: Fire detected at (8, 9)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (48.6% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 71] 🚁 DRONE ALERT: Fire detected at (7, 7)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (49.5% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 71] 🚁 DRONE ALERT: Fire detected at (8, 7)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (50.1% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 80] 🚁 DRONE ALERT: Fire detected at (4, 6)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (49.4% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 81] 🚁 DRONE ALERT: Fire detected at (7, 6)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (49.8% confidence)
   🤖 SYSTEM COMMAND: Rerouting Rescue Agent to avoid coordinates.

[Step 85] 🚁 DRONE ALERT: Fire detected at (3, 6)
   ⚠️ NLP CLASSIFICATION: Lethal Hazard (50.4% confidence)
   🤖 SYSTEM COMM